# Solcellefeltet: fra solretning til en felles DC-buss

## Øst-, sør- og vestvendte solcellefelt, Kirchhoffs lover og koblede ODE-er

### Pilotprosjekt for Matematikk 1

Et bygg har tre solcellefelt:

- ett østvendt felt,
- ett sørvendt felt,
- ett vestvendt felt.

Feltene får ulik solinnstråling gjennom dagen, men leverer energi til en felles DC-buss og last. I et virkelig anlegg vil feltene normalt kobles gjennom effektelektronikk og egne reguleringsfunksjoner. Her bruker vi en pedagogisk kretsmodell som følger progresjonen i det parallelle emnet i elektriske kretser.

Prosjektet har fire deler:

1. **Solgeometri og resistivt DC-nett:** panelnormaler, skalarprodukt, Kirchhoffs lover og et lineært system
2. **Dioder og busskondensator:** stykkevis strømfordeling og en skalar ODE
3. **Induktive grener og felles DC-buss:** et koblet vektor-ODE-system
4. **Felles- og differansemoder:** ortogonalt variabelbytte, egenverdier og et RLC-lignende delsystem

### Læringsmål

Etter prosjektet skal du kunne

- representere sol- og panelretninger med enhetsvektorer,
- bruke en matrise til å beregne innstråling på flere takflater,
- sette opp Kirchhoffs spennings- og strømlover som $Ax=b$,
- kontrollere strøm- og effektbalanse,
- modellere en ideell blokkeringsdiode med en stykkevis funksjon,
- løse en lineær førsteordens ODE analytisk i et fast diodeintervall,
- bruke Euler på et stykkevis system,
- skrive en krets med tre spoler og én kondensator som en vektor-ODE,
- gjennomføre et ortogonalt variabelbytte,
- tolke fellesstrøm og ubalansestrømmer,
- analysere et $2\times2$-system med egenverdier.

### Avgrensning

Solcellefeltene beskrives ikke med en full én-diodemodell eller svitsjende DC–DC-omformere. Kildespenningene er pedagogiske lokalmodeller. Del C kan tolkes som en lineær gjennomsnittsmodell rundt et valgt driftspunkt.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Anlegget og koordinatsystemet

Vi bruker aksene

- $x$: øst,
- $y$: nord,
- $z$: opp.

Solretningen er en enhetsvektor

$$
s(t)=
\begin{pmatrix}s_x(t)\\s_y(t)\\s_z(t)\end{pmatrix},
\qquad \|s(t)\|=1.
$$

Panelnormalen for en flate med helning $\beta$ fra horisontalplanet og asimut $\gamma$ måles her med

- nord: $0^\circ$,
- øst: $90^\circ$,
- sør: $180^\circ$,
- vest: $270^\circ$.

Da bruker vi

$$
\boxed{
n(\beta,\gamma)=
\begin{pmatrix}
\sin\beta\sin\gamma\\
\sin\beta\cos\gamma\\
\cos\beta
\end{pmatrix}.}
$$

In [ ]:
def panelnormal(helning, asimut):
    return np.array([
        np.sin(helning)*np.sin(asimut),
        np.sin(helning)*np.cos(asimut),
        np.cos(helning)
    ])

helning = np.deg2rad(35.0)

n_øst = panelnormal(helning, np.deg2rad(90.0))
n_sør = panelnormal(helning, np.deg2rad(180.0))
n_vest = panelnormal(helning, np.deg2rad(270.0))

N_panel = np.vstack([n_øst, n_sør, n_vest])
navn = ["øst", "sør", "vest"]

print("Panelnormaler:
", N_panel)
print("Normer:", np.linalg.norm(N_panel, axis=1))

# Del A: Solgeometri og resistivt DC-nett

## A.1 Solretning som gitt data

Vi bruker en forenklet dagsmodell. Solens høyde og asimut gis direkte som funksjoner av tiden. Dette er ikke en astronomisk modell for et bestemt sted og dato.

Solretningen beregnes fra solhøyden $\alpha$ og solasimutten $\gamma_s$:

$$
\boxed{
s=
\begin{pmatrix}
\cos\alpha\sin\gamma_s\\
\cos\alpha\cos\gamma_s\\
\sin\alpha
\end{pmatrix}.}
$$

Den direkte normalstrålingen betegnes $DNI$.

In [ ]:
def soldata(t_timer):
    """Pedagogisk klarværsdag mellom kl. 6 og 18."""
    t = np.asarray(t_timer, dtype=float)
    fase = np.pi*(t - 6.0)/12.0

    høyde = np.where(
        (t >= 6.0) & (t <= 18.0),
        np.deg2rad(55.0)*np.sin(fase),
        0.0
    )

    # Øst 90 grader ved soloppgang, sør 180 ved middag, vest 270 ved solnedgang.
    asimut = np.deg2rad(90.0 + 15.0*(t - 6.0))

    s = np.column_stack([
        np.cos(høyde)*np.sin(asimut),
        np.cos(høyde)*np.cos(asimut),
        np.sin(høyde)
    ])

    dagslys = (t >= 6.0) & (t <= 18.0)
    DNI = np.where(dagslys, 850.0*np.maximum(np.sin(fase), 0.0)**0.35, 0.0)
    DHI = np.where(dagslys, 110.0, 0.0)
    GHI = DNI*np.maximum(np.sin(høyde), 0.0) + DHI

    return s, DNI, DHI, GHI

## Oppgave A1: Innstråling på tre orienteringer

Skalarproduktene for alle tre feltene beregnes samtidig:

$$
\boxed{c(t)=N_{panel}s(t).}
$$

Negative verdier erstattes med null. Direkte stråling på feltet blir

$$G_{dir}=DNI[c]_+.$$

Vi legger til en enkel isotrop diffus komponent og bakkereflektert komponent:

$$G_{diff,i}=DHI\frac{1+\cos\beta_i}{2},$$

$$G_{bakke,i}=\rho_gGHI\frac{1-\cos\beta_i}{2}.$$

Beregn den samlede innstrålingsvektoren ved kl. 8, 12 og 16.

In [ ]:
albedo = 0.20
helninger = np.full(3, helning)


def innstråling_paneler(t_timer):
    s, DNI, DHI, GHI = soldata(np.atleast_1d(t_timer))

    cosinus = s @ N_panel.T
    cosinus_pos = np.maximum(cosinus, 0.0)

    G_dir = DNI[:, None]*cosinus_pos
    G_diff = DHI[:, None]*(1 + np.cos(helninger))[None, :]/2
    G_bakke = albedo*GHI[:, None]*(1 - np.cos(helninger))[None, :]/2

    return G_dir + G_diff + G_bakke

for tid in [8.0, 12.0, 16.0]:
    G_poa = ...
    print(f"Kl. {tid:04.1f}:", G_poa, "W/m^2")

## A.2 Fra solinnstråling til kildespenning

Hvert solcellefelt beskrives med en lokal lineær kildekarakteristikk:

$$
\boxed{v_k=V_{oc,k}-r_ki_k.}
$$

Her er

- $V_{oc,k}$ en lysavhengig kildespenning,
- $r_k$ en effektiv indre motstand,
- $i_k$ strømmen fra feltet.

Vi bruker den pedagogiske sammenhengen

$$
V_{oc,k}=V_{mørk}+a_GG_k.
$$

Denne sammenhengen er ikke en full PV-modell. Den brukes for å koble solgeometrien til kretsanalysen.

In [ ]:
V_mørk = 160.0
spenningskoeff = 0.055  # V per W/m^2
r_indre = np.array([1.8, 1.8, 1.8])
R_last = 18.0


def kildespenninger(t_timer):
    G = innstråling_paneler(t_timer)
    return V_mørk + spenningskoeff*G

print("Kildespenninger kl. 12:", kildespenninger(12.0)[0])

## A.3 Kirchhoffs lover som et lineært system

Alle tre felt er koblet til en felles buss med spenning $v$. Uten dioder gjelder

$$r_ki_k+v=V_{oc,k}.$$

Kirchhoffs strømlov gir

$$i_1+i_2+i_3=\frac{v}{R_L}.$$

Dermed får vi

$$
\boxed{
\begin{pmatrix}
r_1&0&0&1\\
0&r_2&0&1\\
0&0&r_3&1\\
1&1&1&-1/R_L
\end{pmatrix}
\begin{pmatrix}i_1\\i_2\\i_3\\v\end{pmatrix}
=
\begin{pmatrix}V_{oc,1}\\V_{oc,2}\\V_{oc,3}\\0\end{pmatrix}.}
$$

## Oppgave A2: Løs strømfordelingen

Implementer systemet og løs det ved kl. 8, 12 og 16. Kontroller residualet.

In [ ]:
def resistiv_løsning(Voc, r=r_indre, R_last=R_last):
    A = np.array([
        [r[0], 0.0,  0.0, 1.0],
        [0.0,  r[1], 0.0, 1.0],
        [0.0,  0.0,  r[2], 1.0],
        [1.0,  1.0,  1.0, -1.0/R_last]
    ])
    b = np.append(Voc, 0.0)
    x = ...
    return x, A, b

for tid in [8.0, 12.0, 16.0]:
    Voc = kildespenninger(tid)[0]
    x, A, b = resistiv_løsning(Voc)
    print("
Tid:", tid)
    print("[i_øst, i_sør, i_vest, v_buss] =", x)
    print("Residualnorm =", ...)

## Oppgave A3: Effektbalanse

Beregn

$$P_{kilde,k}=V_{oc,k}i_k,$$

$$P_{tap,k}=r_ki_k^2,$$

og

$$P_{last}=\frac{v^2}{R_L}.$$

Kontroller

$$
\sum_kP_{kilde,k}
\approx
\sum_kP_{tap,k}+P_{last}.
$$

Tolk eventuelle negative grenstrømmer. Hvorfor er de et tegn på at modellen trenger blokkeringsdioder eller effektelektronikk?

In [ ]:
tid = 8.0
Voc = kildespenninger(tid)[0]
x, _, _ = resistiv_løsning(Voc)
i = x[:3]
v_buss = x[3]

P_kilder = ...
P_tap = ...
P_last = ...

print("Kildeeffekter:", P_kilder)
print("Tap:", P_tap)
print("Lasteffekt:", P_last)
print("Effektbalansefeil:", ...)

# Del B: Blokkeringsdioder og busskondensator

## B.1 En ideell diodefunksjon

Vi modellerer grenstrømmen som

$$
\boxed{
i_k(v,t)=
\max\left(0,\frac{V_{oc,k}(t)-v}{r_k}\right).}
$$

Hvis kildespenningen er lavere enn busspenningen, blokkerer dioden tilbakestrømmen.

Dette er en ideell og stykkevis modell. Virkelige dioder har spenningsfall og mer komplisert strøm-spenningskarakteristikk.

In [ ]:
def diode_strømmer(v_buss, t_timer):
    Voc = kildespenninger(t_timer)[0]
    return np.maximum(0.0, (Voc - v_buss)/r_indre)

for v in [170.0, 190.0, 210.0]:
    print("v =", v, "V, strømmer kl. 8 =", diode_strømmer(v, 8.0))

## B.2 Stasjonær diodekobling

Ved likevekt må

$$
\sum_ki_k(v)=\frac{v}{R_L}.
$$

Siden de ledende diodene ikke er kjent på forhånd, kan vi finne likevekten med et enkelt rutenettsøk eller en ferdig lærerfunksjon.

I hovedoppgaven brukes rutenettsøk. Dette er ikke ment som generell rotfinningsundervisning.

In [ ]:
def diode_likevekt(t_timer, v_min=0.0, v_max=260.0, antall=20001):
    v_grid = np.linspace(v_min, v_max, antall)
    rest = np.array([
        np.sum(diode_strømmer(v, t_timer)) - v/R_last
        for v in v_grid
    ])
    indeks = np.argmin(np.abs(rest))
    v = v_grid[indeks]
    return v, diode_strømmer(v, t_timer), rest[indeks]

for tid in [8.0, 12.0, 16.0]:
    v, i, rest = diode_likevekt(tid)
    print(tid, v, i, "KCL-rest:", rest)

## B.3 Kondensator på DC-bussen

Med en busskondensator blir Kirchhoffs strømlov

$$
\boxed{
C\dot v
=
\sum_{k=1}^3i_k(v,t)-\frac{v}{R_L}.}
$$

Dette er en skalar, stykkevis ikke-lineær ODE.

Kondensatoren lagrer energi

$$E_C=\frac12Cv^2$$

og kan derfor dempe raske endringer i busspenningen.

In [ ]:
C_buss = 0.18  # F


def skyfaktor(t_s):
    """En sky reduserer sørfeltet mellom 4 og 7 sekunder."""
    return 0.55 if 4.0 <= t_s < 7.0 else 1.0


def kildespenninger_transient(t_s, klokkeslett=12.0):
    Voc = kildespenninger(klokkeslett)[0].copy()
    # Reduser den solavhengige delen, ikke mørkespenningen.
    Voc[1] = V_mørk + skyfaktor(t_s)*(Voc[1] - V_mørk)
    return Voc


def diode_strømmer_transient(v_buss, t_s):
    Voc = kildespenninger_transient(t_s)
    return np.maximum(0.0, (Voc - v_buss)/r_indre)


def buss_ode(t_s, v_buss):
    i = diode_strømmer_transient(v_buss, t_s)
    return (np.sum(i) - v_buss/R_last)/C_buss

## Oppgave B1: Håndløsning i ett diodeintervall

Når et fast sett av dioder leder og kildespenningene er konstante, blir ligningen lineær:

$$
C\dot v
=
\sum_{k\in\mathcal L}\frac{V_{oc,k}}{r_k}
-
\left(
\sum_{k\in\mathcal L}\frac1{r_k}
+\frac1{R_L}
\right)v.
$$

Skriv den som

$$\dot v+av=b.$$

Finn likevektsspenningen og tidskonstanten når alle tre dioder leder ved middag uten sky. Kontroller deretter at alle grenstrømmene faktisk er positive ved den beregnede likevekten.

In [ ]:
Voc_middag = kildespenninger_transient(0.0)

sum_ledning = ...
a = ...
b = ...
v_likevekt = ...
tau = ...

print("Likevektsspenning:", v_likevekt)
print("Tidskonstant:", tau)
print("Grenstrømmer:", (Voc_middag-v_likevekt)/r_indre)

## Oppgave B2: Euler gjennom en skyggehendelse

Simuler busspenningen fra 0 til 12 sekunder. Start i likevekt før skyen kommer.

Plott

- busspenningen,
- de tre grenstrømmene,
- hvilke dioder som leder.

Gjenta med en mindre og en større kondensator.

In [ ]:
def euler_skalar(f, y0, sluttid, h):
    n = int(round(sluttid/h))
    t = np.linspace(0.0, n*h, n + 1)
    y = np.zeros(n + 1)
    y[0] = y0

    for k in range(n):
        y[k + 1] = ...

    return t, y


t_B, v_B = euler_skalar(buss_ode, v_likevekt, 12.0, 0.002)
i_B = np.array([diode_strømmer_transient(v, t) for t, v in zip(t_B, v_B)])

fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(t_B, v_B)
ax[0].set_ylabel("Busspenning V")
ax[0].grid()

for j, navn_j in enumerate(navn):
    ax[1].plot(t_B, i_B[:, j], label=navn_j)
ax[1].set_xlabel("Tid s")
ax[1].set_ylabel("Grenstrøm A")
ax[1].legend()
ax[1].grid()
plt.show()

# Del C: Induktive grener og felles DC-buss

## C.1 Lineær gjennomsnittsmodell

Vi legger inn én induktans i hver inngangsgren. Dette kan tolkes som en forenklet modell av spoler og filtre i tre DC–DC-grener.

Tilstanden er

$$
\boxed{
x=
\begin{pmatrix}i_E\\i_S\\i_V\\v\end{pmatrix}.}
$$

Kretsligningene er

$$
\boxed{
\begin{aligned}
L_1\dot i_E&=V_E(t)-R_1i_E-v,\\
L_2\dot i_S&=V_S(t)-R_2i_S-v,\\
L_3\dot i_V&=V_V(t)-R_3i_V-v,\\
C\dot v&=i_E+i_S+i_V-\frac{v}{R_L}.
\end{aligned}}
$$

I hovedmodellen representerer grenene aktive omformere eller en lokal lineær modell. Vi bruker derfor ikke diodeavkuttingen i del C.

## C.2 Matriseform

Skriv

$$
\boxed{M\dot x=Ax+f(t),}
$$

med

$$
M=
\begin{pmatrix}
L_1&0&0&0\\
0&L_2&0&0\\
0&0&L_3&0\\
0&0&0&C
\end{pmatrix},
$$

$$
A=
\begin{pmatrix}
-R_1&0&0&-1\\
0&-R_2&0&-1\\
0&0&-R_3&-1\\
1&1&1&-1/R_L
\end{pmatrix},
$$

og

$$
f(t)=
\begin{pmatrix}V_E(t)\\V_S(t)\\V_V(t)\\0\end{pmatrix}.
$$

In [ ]:
L_gren = np.array([0.08, 0.08, 0.08])
R_gren = np.array([1.8, 1.8, 1.8])
C_dc = 0.18

M = np.diag([L_gren[0], L_gren[1], L_gren[2], C_dc])
A_sys = np.array([
    [-R_gren[0], 0.0, 0.0, -1.0],
    [0.0, -R_gren[1], 0.0, -1.0],
    [0.0, 0.0, -R_gren[2], -1.0],
    [1.0, 1.0, 1.0, -1.0/R_last]
])


def pådrag(t_s):
    return np.append(kildespenninger_transient(t_s), 0.0)


def dc_ode(t_s, x):
    return ...

## Oppgave C1: Stasjonær starttilstand

Før skyen kommer er $f(t)$ konstant. Likevekten oppfyller

$$0=Ax^*+f.$$

Finn starttilstanden med et lineært system. Kontroller residualet.

In [ ]:
x_likevekt = ...
print("Likevekt [iE, iS, iV, v]:", x_likevekt)
print("Residual:", ...)

## Oppgave C2: Euler for vektorsystemet

Simuler den samme skyggehendelsen som i del B. Sammenlign busspenning og grenstrømmer med kondensatormodellen uten induktanser.

In [ ]:
def euler_system(f, x0, sluttid, h):
    n = int(round(sluttid/h))
    t = np.linspace(0.0, n*h, n + 1)
    X = np.zeros((n + 1, len(x0)))
    X[0] = x0

    for k in range(n):
        X[k + 1] = ...

    return t, X


t_C, X_C = euler_system(dc_ode, x_likevekt, 12.0, 0.0005)

fig, ax = plt.subplots(2, 1, sharex=True)
for j, navn_j in enumerate(navn):
    ax[0].plot(t_C, X_C[:, j], label=navn_j)
ax[0].set_ylabel("Grenstrøm A")
ax[0].legend()
ax[0].grid()

ax[1].plot(t_C, X_C[:, 3])
ax[1].set_xlabel("Tid s")
ax[1].set_ylabel("Busspenning V")
ax[1].grid()
plt.show()

## Oppgave C3: Endre kretsparametrene

Undersøk virkningen av

- større og mindre induktans,
- større og mindre busskondensator,
- større og mindre lastmotstand,
- større tap i grenene.

Sammenlign oversving, innstillingstid og maksimal strømendring.

# Del D: Felles- og differansemoder

Når alle tre grener har samme $L$ og $R$, kan strømvektoren

$$i=(i_E,i_S,i_V)^T$$

beskrives med tre nye koordinater.

Vi velger de ortonormale vektorene

$$
q_m=\frac1{\sqrt3}
\begin{pmatrix}1\\1\\1\end{pmatrix},
$$

$$
q_d=\frac1{\sqrt2}
\begin{pmatrix}1\\0\\-1\end{pmatrix},
$$

$$
q_c=\frac1{\sqrt6}
\begin{pmatrix}1\\-2\\1\end{pmatrix}.
$$

Samle dem som kolonner i

$$Q=(q_m\ q_d\ q_c).$$

De nye strømkoordinatene er

$$
\boxed{y=Q^Ti.}
$$

Her er

- $i_m$: fellesstrømmen,
- $i_d$: øst–vest-ubalansen,
- $i_c$: sørfeltet mot øst- og vestfeltet.

## Oppgave D1: Kontroller variabelbyttet

Bygg $Q$ og kontroller

$$Q^TQ=I,$$

$$QQ^T=I.$$

Transformer noen strømvektorer og tolk resultatene.

In [ ]:
q_m = np.array([1.0, 1.0, 1.0])/np.sqrt(3)
q_d = np.array([1.0, 0.0, -1.0])/np.sqrt(2)
q_c = np.array([1.0, -2.0, 1.0])/np.sqrt(6)
Q = np.column_stack([q_m, q_d, q_c])

print("Q^T Q:
", ...)
print("Q Q^T:
", ...)

for i_test in [
    np.array([5.0, 5.0, 5.0]),
    np.array([6.0, 5.0, 4.0]),
    np.array([5.0, 2.0, 5.0])
]:
    print("i =", i_test, "gir y =", ...)

## D.2 Pådraget i de nye koordinatene

Kildespenningene transformeres på samme måte:

$$
\begin{pmatrix}V_m\\V_d\\V_c\end{pmatrix}
=Q^T
\begin{pmatrix}V_E\\V_S\\V_V\end{pmatrix}.
$$

- Lik spenning fra alle felt gir bare fellespådrag.
- Forskjell mellom øst og vest aktiverer øst–vest-moden.
- Et sørfelt som avviker fra de to andre aktiverer den tredje moden.

## Oppgave D2: Solretning og moder

Beregn de tre pådragskoordinatene kl. 8, 12 og 16. Forklar hvilke moder som dominerer.

In [ ]:
for tid in [8.0, 12.0, 16.0]:
    Voc = kildespenninger(tid)[0]
    V_modal = ...
    print(tid, Voc, "->", V_modal)

# D.3 Frakobling av strømforskjellene

For like grenparametre får differansemodene

$$
\boxed{L\dot i_d=-Ri_d+V_d(t),}
$$

$$
\boxed{L\dot i_c=-Ri_c+V_c(t).}
$$

Busspenningen er ikke med, fordi differansevektorene har komponentsum null.

Fellesmoden og busspenningen danner derimot systemet

$$
\boxed{
\begin{aligned}
L\dot i_m&=-Ri_m-\sqrt3v+V_m(t),\\
C\dot v&=\sqrt3i_m-\frac{v}{R_L}.
\end{aligned}}
$$

## Oppgave D3: Verifiser den modale modellen numerisk

Transformer strømhistorikken fra del C:

$$y(t)=Q^Ti(t).$$

Plott $i_m,i_d,i_c$ og busspenningen. Hvilken mode aktiveres mest når bare sørfeltet skygges?

In [ ]:
i_modal = ...

plt.plot(t_C, i_modal[:, 0], label="fellesmode")
plt.plot(t_C, i_modal[:, 1], label="øst-vest")
plt.plot(t_C, i_modal[:, 2], label="sør mot kantfelt")
plt.xlabel("Tid s")
plt.ylabel("Modal strøm")
plt.grid()
plt.legend()
plt.show()

## D.4 Egenverdier til RLC-delsystemet

Det homogene $2\times2$-systemet for $(i_m,v)$ har systemmatrisen

$$
\boxed{
A_m=
\begin{pmatrix}
-R/L&-\sqrt3/L\\
\sqrt3/C&-1/(R_LC)
\end{pmatrix}.}
$$

Egenverdiene bestemmer om transienten er oscillerende eller ikke, og hvor raskt den dempes.

## Oppgave D4: Egenverdier og diagonalisering

1. Beregn egenverdiene til $A_m$.
2. Sammenlign dem med responsen i del C.
3. Undersøk hvordan egenverdiene endres når $L$, $C$ eller $R$ endres.
4. Dersom matrisen har to ulike egenverdier, kontroller en diagonalisering
   
   $$A_m=PDP^{-1}.$$

In [ ]:
L0 = L_gren[0]
R0 = R_gren[0]

A_modal = np.array([
    [-R0/L0, -np.sqrt(3)/L0],
    [ np.sqrt(3)/C_dc, -1/(R_last*C_dc)]
])

egenverdier, P = ...
D = ...

print("A_modal:
", A_modal)
print("Egenverdier:", egenverdier)
print("Egenvektorer:
", P)
print("Kontroll A=PDP^-1:
", ...)

## Oppgave D5: Direkte mot modal simulering

Implementer de to differanse-ODE-ene og det koblede $2\times2$-systemet. Transformer deretter tilbake:

$$i=Qy.$$

Sammenlign med den direkte firetilstandssimuleringen fra del C.

In [ ]:
# Implementer modal simulering og sammenlign maksimal forskjell.

# Modellkritikk

Diskuter minst fem punkter:

- Solstillingen er en pedagogisk funksjon, ikke en lokalisert astronomisk modell.
- Diffus og bakkereflektert stråling behandles enkelt.
- Kildespenningen er lineær i innstrålingen.
- Den resistive modellen er bare lokal og inkluderer ikke hele PV-kurven.
- Diodene i del B er ideelle.
- Rutenettsøket for diodelikevekt er en enkel hjelpealgoritme.
- Del C bruker en lineær gjennomsnittsmodell, ikke svitsjende effektelektronikk.
- MPPT-regulering er utelatt.
- Grenparametrene antas like i modalanalysen.
- Virkelige omformere har tap, metning, begrensninger og styringssløyfer.
- Temperaturens virkning på PV-spenning og effekt er utelatt i hovedmodellen.
- Euler kan kreve svært liten steglengde når $L$ eller $C$ er liten.
- En ekte serie- eller parallellkobling av paneler kan reagere annerledes ved delvis skygge.

## Mulige videreføringer

- én-diodemodell for hvert solcellefelt,
- temperaturavhengige kildeparametre,
- svitsjende eller middelverdibasert boost-omformer,
- MPPT-regulering,
- batteri på DC-bussen,
- inverter og AC-last,
- lokale værdata og solstilling,
- sammenligning med måledata fra et virkelig anlegg.

# Oppsummering

Skriv en kort rapport der du forklarer

1. hvordan solvektoren og panelnormalene bestemte innstrålingen,
2. hvordan Kirchhoffs lover ga et $4\times4$-system,
3. hvorfor negative grenstrømmer viste behov for dioder eller omformere,
4. hvordan busskondensatoren ga en skalar ODE,
5. når håndløsningen for kondensator-ODE-en var gyldig,
6. hvordan spoler og kondensator ga et firekomponents vektor-ODE-system,
7. hvordan skyen over sørfeltet påvirket strømmer og busspenning,
8. hva felles- og differansemodene betyr fysisk,
9. hvorfor differansemodene frakobles busspenningen,
10. hva egenverdiene til RLC-delsystemet forteller om transienten.

## Referanser for prosjektutviklingen

Prosjektet er inspirert av standard modeller for solinnstråling i panelplanet, PV-ekvivalentkretser og tilstandsrommodeller for DC–DC-omformere. Studentene trenger ikke lese eksterne kilder for å gjennomføre prosjektet.